# RemasterPhantom — Llama-3.2-1B-Instruct LoRA 파인튜닝

파일 시그니처 무결성 검증, 카나리 기반 암호화/복호화, 랜섬웨어 대응, MASTER CANARY/TLS 보호를 수행하는 방어 전문 LLM 에이전트를 학습시킨다.

**예상 시간 (Apple M2 · 8GB · MPS 기준)**
| 항목 | 예상 시간 |
|---|---|
| 베이스 모델 다운로드 (2.5GB) | 5–15분 |
| 데이터 로드/전처리 | 1–2분 |
| 학습 1 epoch (5,880 예제, bs=1×accum8, seq 1024) | 약 40–70분 |
| 학습 3 epoch (권장) | 약 2–3.5시간 |
| 어댑터 병합·납출 | 약 5분 |

> GPU(CUDA) 환경이면 3 epoch 기준 20–40분이면 충분하다. M2 8GB에서는 메모리가 총량이므로 다른 대용량 앱을 닫고 실행 권장.

## 1. 설정

베이스 모델은 게이트드 저장소다. HF 로그인 + 라이선스 동의가 필요하다.
미동의 상태라면 아래 `BASE_MODEL`을 비게이트 미러(`unsloth/Llama-3.2-1B-Instruct`)로 바꿔 실행하면 된다.

In [ ]:
import os
import torch

# --- 인증 (터미널에서 `hf auth login` 또는 아래 직접 로그인) ---
# from huggingface_hub import login; login("hf_xxxxxxxxxxxxxxxx")

# 베이스 모델: 게이트드 meta 모델 or 비게이트 미러
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
# BASE_MODEL = "unsloth/Llama-3.2-1B-Instruct"   # 라이선스 미동의 시 주석 해제

DATA_PATH = "../data/remasterphantom_sft.jsonl"
OUTPUT_DIR = "../outputs/remasterphantom-lora"
MERGED_DIR = "../outputs/RemasterPhantom-merged"

MAX_LENGTH = 1024
EPOCHS = 3
LR = 2e-4
BATCH_SIZE = 1          # M2 8GB는 배치 1 권장
GRAD_ACCUM = 8          # 유효 배치 8
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device in ("mps", "cuda") else torch.float32
print(f"device={device}, dtype={dtype}, torch={torch.__version__}")

## 2. 토크나이저·모델 로드

8GB RAM 환경에 맞춰 gradient checkpointing과 캐시 비활성화를 켠다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = model.to(device)
print("params:", sum(p.numel() for p in model.parameters()) / 1e9, "B")

## 3. 데이터셋 로드

`generate_dataset.py`로 생성한 5,880개 지시-응답 쌍. `messages` 형식이므로 trl이 낶은 템플릿으로 자동 처리된다.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=DATA_PATH, split="train")
ds = ds.shuffle(seed=42)
split = ds.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(train_ds)
print(eval_ds)
print(train_ds[0]["messages"][1]["content"][:120])

## 4. LoRA 어댑터 구성

attention + MLP 전체 선형층에 어댑터를 장착해 표현력을 확보한다 (정확도 우선).

In [ ]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 5. 학습

trl `SFTTrainer`로 전체 시퀀스 손실 방식 SFT를 수행한다.
(베이스 챗 템플릿에 `{% generation %}` 마커가 없어 assistant-only 손실은 사용하지 않는다.)

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=60,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=(dtype == torch.bfloat16),
    max_length=MAX_LENGTH,
    packing=False,
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
    dataset_num_proc=4,
)

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)
trainer.train()

## 6. 어댑터 저장 + 베이스 병합

배포용으로는 병합 모델을 납출한다 (추론 시 peft 없이 바로 로드 가능).

In [ ]:
trainer.save_model(OUTPUT_DIR)          # 어댑터 + 토크나이저
tokenizer.save_pretrained(OUTPUT_DIR)

# 병합
merged = model.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("saved:", MERGED_DIR)

## 7. 추론 스모크 테스트

학습된 모델이 시그니처 질문과 사고 대응을 제대로 학습했는지 확인한다.

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model=MERGED_DIR, tokenizer=MERGED_DIR, device_map=device)
tests = [
    "hex dump 시작이 89504E470D0A1A0A입니다. 어떤 파일인가요?",
    "카나리 무결성이 물너졌다는 신호는 무엇이며, 그때 해야 할 일은?",
    "MASTER CANARY는 무엇이고 파일의 TLS를 어떻게 보호하나요?",
]
for q in tests:
    msgs = [{"role": "user", "content": q}]
    out = gen(msgs, max_new_tokens=220, do_sample=False)
    print("Q:", q)
    print("A:", out[0]["generated_text"][-1]["content"])
    print("-" * 80)

## 8. HuggingFace 업로드

1. 저장소 생성: `k4zt0/RemasterPhantom`  
2. 병합 모델 + 어댑터 + 데이터셋 카드 업로드  
실행 전 `hf auth login` 또는 위 셀의 `login()`으로 인증 필요.

In [ ]:
from huggingface_hub import HfApi, create_repo, whoami

try:
    user = whoami()["name"]
except Exception as e:
    raise RuntimeError("HF 로그인이 필요합니다: hf auth login") from e

REPO_ID = f"{user}/RemasterPhantom"
api = HfApi()
create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path=MERGED_DIR, repo_id=REPO_ID, commit_message="RemasterPhantom v0.1 — merged model")
print("uploaded:", f"https://huggingface.co/{REPO_ID}")